<a href="https://colab.research.google.com/github/aakilkhanemon/AI-driven-marine-lead-discovery/blob/main/gnina_3l4q_x_single_ligand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!wget https://github.com/gnina/gnina/releases/download/v1.1/gnina -O gnina
!chmod +x gnina

--2026-05-24 01:22:32--  https://github.com/gnina/gnina/releases/download/v1.1/gnina
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/45548146/bc227ff8-7934-457d-95b3-eab58982638a?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-24T02%3A02%3A27Z&rscd=attachment%3B+filename%3Dgnina&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-24T01%3A02%3A16Z&ske=2026-05-24T02%3A02%3A27Z&sks=b&skv=2018-11-09&sig=3XZwXZGQehdKwicYyzj5jCtaibkJ%2FP23Wj4c%2FwIylgw%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3OTU4OTM1MiwibmJmIjoxNzc5NTg1NzUyLCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmNvcmUud2luZG93cy5uZXQif

In [8]:
import numpy as np
import math

# 1. Ensure structural analysis toolkit is available
try:
    import MDAnalysis as mda
except ImportError:
    !pip install -q mdanalysis
    import MDAnalysis as mda

# 2. Assign your specific target parameters
PDB_FILE = "receptor.pdb"  # Ensure your MOE-prepped file matches this name
TARGET_RESIDUES = "89 96 97 164"

try:
    u = mda.Universe(PDB_FILE)

    # Target your specific residues strictly on Chain A using alpha carbons & backbone atoms
    selection_string = f"chainID A and resid {TARGET_RESIDUES} and (name CA C N O or backbone)"
    pocket_atoms = u.select_atoms(selection_string)

    # Fallback protocol if the PDB uses alternative chain naming conventions
    if len(pocket_atoms) == 0:
        selection_string = f"resid {TARGET_RESIDUES}"
        pocket_atoms = u.select_atoms(selection_string)
        print("[Warning] Clean Chain A attributes not found. Initialized global residue selection fallback.")

    # 3. Calculate Spatial Parameters
    center = pocket_atoms.center_of_mass()
    positions = pocket_atoms.positions
    coordinate_span = np.max(positions, axis=0) - np.min(positions, axis=0)

    # Apply a standard padding metric to accommodate ligand rotational freedom
    suggested_box_dim = np.max(coordinate_span) + 8.0
    final_box_size = max(18.0, math.ceil(suggested_box_dim))

    # 4. Generate the Automated GNINA Execution String
    print("\n" + "="*60)
    print(" TARGET POCKET MAPPING COMPLETE")
    print("="*60)
    print(f"Target Residues Successfully Mapped: Tyr89, Glu96, Glu97, Pro164")
    print(f"Calculated Center : X = {center[0]:.3f} | Y = {center[1]:.3f} | Z = {center[2]:.3f}")
    print(f"Cubic Box Size    : {final_box_size} Ångströms")
    print("-"*60)
    print("EXECUTE THIS COMMAND IN YOUR NEXT COLAB CELL TO START DOCKING:")
    print(f"./gnina -r {PDB_FILE} -l ligand.sdf \\")
    print(f"        --center_x {center[0]:.3f} --center_y {center[1]:.3f} --center_z {center[2]:.3f} \\")
    print(f"        --size_x {final_box_size} --size_y {final_box_size} --size_z {final_box_size} \\")
    print(f"        --num_modes 5 --device cuda --cnn_scoring rescore --out test_docked.sdf")
    print("="*60 + "\n")

except Exception as e:
    print(f"[Error] Execution failed: {e}")
    print("Verify your PDB file is uploaded and that the residue numbers 89, 96, 97, and 164 exist in the file.")


 TARGET POCKET MAPPING COMPLETE
Target Residues Successfully Mapped: Tyr89, Glu96, Glu97, Pro164
Calculated Center : X = -24.980 | Y = -2.037 | Z = -27.028
Cubic Box Size    : 29 Ångströms
------------------------------------------------------------
EXECUTE THIS COMMAND IN YOUR NEXT COLAB CELL TO START DOCKING:
./gnina -r receptor.pdb -l ligand.sdf \
        --center_x -24.980 --center_y -2.037 --center_z -27.028 \
        --size_x 29 --size_y 29 --size_z 29 \
        --num_modes 5 --device cuda --cnn_scoring rescore --out test_docked.sdf



In [14]:
!./gnina -r receptor.pdb -l ligand.sdf \
        --center_x -24.980 --center_y -2.037 --center_z -27.028 \
        --size_x 29 --size_y 29 --size_z 29 \
        --num_modes 5 --device cuda --cnn_scoring rescore --out test_docked.sdf

Command line parse error: the argument ('cuda') for option '--device' is invalid

Correct usage:

Input:
  -r [ --receptor ] arg              rigid part of the receptor
  --flex arg                         flexible side chains, if any (PDBQT)
  -l [ --ligand ] arg                ligand(s)
  --flexres arg                      flexible side chains specified by comma 
                                     separated list of chain:resid
  --flexdist_ligand arg              Ligand to use for flexdist
  --flexdist arg                     set all side chains within specified 
                                     distance to flexdist_ligand to flexible
  --flex_limit arg                   Hard limit for the number of flexible 
                                     residues
  --flex_max arg                     Retain at at most the closest flex_max 
                                     flexible residues

Search space (required):
  --center_x arg                     X coordinate of the center
  --c

In [10]:
!grep -E "CNNscore|CNNaffinity|minimizedAffinity" test_docked.sdf

grep: test_docked.sdf: No such file or directory


In [11]:
!./gnina -r receptor.pdb -l ligand.sdf --center_x -24.980 --center_y -2.037 --center_z -27.028 --size_x 29 --size_y 29 --size_z 29 --num_modes 5 --device cuda --cnn_scoring rescore --out test_docked.sdf

Command line parse error: the argument ('cuda') for option '--device' is invalid

Correct usage:

Input:
  -r [ --receptor ] arg              rigid part of the receptor
  --flex arg                         flexible side chains, if any (PDBQT)
  -l [ --ligand ] arg                ligand(s)
  --flexres arg                      flexible side chains specified by comma 
                                     separated list of chain:resid
  --flexdist_ligand arg              Ligand to use for flexdist
  --flexdist arg                     set all side chains within specified 
                                     distance to flexdist_ligand to flexible
  --flex_limit arg                   Hard limit for the number of flexible 
                                     residues
  --flex_max arg                     Retain at at most the closest flex_max 
                                     flexible residues

Search space (required):
  --center_x arg                     X coordinate of the center
  --c

In [12]:
from rdkit import Chem

# Safely load the completed molecular library from the docking run
supplier = Chem.SDMolSupplier('test_docked.sdf')

for i, mol in enumerate(supplier):
    if mol is None: continue
    print(f"--- Conformation Pose {i+1} ---")
    print(f"Vina Affinity  : {mol.GetProp('minimizedAffinity')} kcal/mol")
    print(f"CNN Match Score: {mol.GetProp('CNNscore')}")
    print(f"CNN Affinity   : {mol.GetProp('CNNaffinity')}\n")

ModuleNotFoundError: No module named 'rdkit'

In [13]:
# 1. Install RDKit quietly in your Colab environment
!pip install -q rdkit

import os
from rdkit import Chem

# 2. Check if the docking output actually exists yet
if not os.path.exists('test_docked.sdf'):
    print("❌ ERROR: 'test_docked.sdf' was not found in your sidebar!")
    print("Please run the single-line GNINA docking command from the previous step first,")
    print("wait for it to complete, refresh your sidebar, and then run this cell again.")
else:
    # 3. Safely parse the structural metadata
    supplier = Chem.SDMolSupplier('test_docked.sdf')
    print("✅ File found! Parsing docking metrics...\n")
    for i, mol in enumerate(supplier):
        if mol is None:
            print(f"Pose {i+1}: Could not parse molecular geometry.")
            continue
        print(f"--- Conformation Pose {i+1} ---")
        print(f"Vina Affinity  : {mol.GetProp('minimizedAffinity')} kcal/mol")
        print(f"CNN Match Score: {mol.GetProp('CNNscore')}")
        print(f"CNN Affinity   : {mol.GetProp('CNNaffinity')}\n")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 56.8 MB/s eta 0:00:00
❌ ERROR: 'test_docked.sdf' was not found in your sidebar!
Please run the single-line GNINA docking command from the previous step first,
wait for it to complete, refresh your sidebar, and then run this cell again.


In [15]:
# 1. Install necessary python structural suites quietly
!pip install -q rdkit mdanalysis

# 2. Fetch and grant execution permissions to the GNINA binary
!wget -q https://github.com/gnina/gnina/releases/download/v1.1/gnina -O gnina
!chmod +x gnina

print("=== ENVIRONMENT SETUP COMPLETE ===")
print("GNINA binary and Python libraries are ready.")

=== ENVIRONMENT SETUP COMPLETE ===
GNINA binary and Python libraries are ready.


In [16]:
!./gnina -r receptor.pdb -l ligand.sdf --center_x -24.980 --center_y -2.037 --center_z -27.028 --size_x 29 --size_y 29 --size_z 29 --num_modes 5 --device cuda --cnn_scoring rescore --out test_docked.sdf

Command line parse error: the argument ('cuda') for option '--device' is invalid

Correct usage:

Input:
  -r [ --receptor ] arg              rigid part of the receptor
  --flex arg                         flexible side chains, if any (PDBQT)
  -l [ --ligand ] arg                ligand(s)
  --flexres arg                      flexible side chains specified by comma 
                                     separated list of chain:resid
  --flexdist_ligand arg              Ligand to use for flexdist
  --flexdist arg                     set all side chains within specified 
                                     distance to flexdist_ligand to flexible
  --flex_limit arg                   Hard limit for the number of flexible 
                                     residues
  --flex_max arg                     Retain at at most the closest flex_max 
                                     flexible residues

Search space (required):
  --center_x arg                     X coordinate of the center
  --c

In [17]:
import os
from rdkit import Chem

output_file = 'test_docked.sdf'

if not os.path.exists(output_file):
    print(f"❌ CRITICAL ERROR: '{output_file}' was never generated by GNINA.")
    print("Please check the terminal logs in Cell 2 for a compilation crash.")
else:
    supplier = Chem.SDMolSupplier(output_file)
    print(f"✅ Successful File Detection! Extracting AI Metrics for {len(supplier)} Poses:\n")

    for i, mol in enumerate(supplier):
        if mol is None:
            print(f"Pose {i+1}: Structural geometry unreadable by RDKit parsing constraints.")
            continue

        print(f"--- Structural Pose Mode {i+1} ---")
        print(f"• Classical Vina Energy : {mol.GetProp('minimizedAffinity')} kcal/mol")
        print(f"• Deep Learning CNN Score: {mol.GetProp('CNNscore')} (Confidence Match)")
        print(f"• Predicted ML Affinity  : {mol.GetProp('CNNaffinity')} (-log Kd scale)\n")

❌ CRITICAL ERROR: 'test_docked.sdf' was never generated by GNINA.
Please check the terminal logs in Cell 2 for a compilation crash.


In [18]:
!./gnina -r receptor.pdb -l ligand.sdf --center_x -24.980 --center_y -2.037 --center_z -27.028 --size_x 29 --size_y 29 --size_z 29 --num_modes 5 --device 0 --cnn_scoring rescore --out test_docked.sdf

Streaming output truncated to the last 5000 lines.
  but OpenBabel found '  ' (atom 458)
*** Open Babel Warning  in parseAtomRecord
  Problems reading a HETATM or ATOM record.
  According to the PDB specification,
  columns 77-78 should contain the element symbol of an atom.
  but OpenBabel found '  ' (atom 459)
*** Open Babel Warning  in parseAtomRecord
  Problems reading a HETATM or ATOM record.
  According to the PDB specification,
  columns 77-78 should contain the element symbol of an atom.
  but OpenBabel found '  ' (atom 460)
*** Open Babel Warning  in parseAtomRecord
  Problems reading a HETATM or ATOM record.
  According to the PDB specification,
  columns 77-78 should contain the element symbol of an atom.
  but OpenBabel found '  ' (atom 461)
*** Open Babel Warning  in parseAtomRecord
  Problems reading a HETATM or ATOM record.
  According to the PDB specification,
  columns 77-78 should contain the element symbol of an atom.
  but OpenBabel found '  ' (atom 462)
*** Open Ba

In [19]:
import os
from rdkit import Chem

output_file = 'test_docked.sdf'

if not os.path.exists(output_file):
    print(f"❌ CRITICAL ERROR: '{output_file}' was never generated by GNINA.")
    print("Please check the terminal logs in Cell 2 for a compilation crash.")
else:
    supplier = Chem.SDMolSupplier(output_file)
    print(f"✅ Successful File Detection! Extracting AI Metrics for {len(supplier)} Poses:\n")

    for i, mol in enumerate(supplier):
        if mol is None:
            print(f"Pose {i+1}: Structural geometry unreadable by RDKit parsing constraints.")
            continue

        print(f"--- Structural Pose Mode {i+1} ---")
        print(f"• Classical Vina Energy : {mol.GetProp('minimizedAffinity')} kcal/mol")
        print(f"• Deep Learning CNN Score: {mol.GetProp('CNNscore')} (Confidence Match)")
        print(f"• Predicted ML Affinity  : {mol.GetProp('CNNaffinity')} (-log Kd scale)\n")

✅ Successful File Detection! Extracting AI Metrics for 5 Poses:

--- Structural Pose Mode 1 ---
• Classical Vina Energy : -4.96274 kcal/mol
• Deep Learning CNN Score: 0.4650994539 (Confidence Match)
• Predicted ML Affinity  : 4.7298851013 (-log Kd scale)

--- Structural Pose Mode 2 ---
• Classical Vina Energy : -4.22238 kcal/mol
• Deep Learning CNN Score: 0.4495382011 (Confidence Match)
• Predicted ML Affinity  : 4.5794839859 (-log Kd scale)

--- Structural Pose Mode 3 ---
• Classical Vina Energy : -4.32232 kcal/mol
• Deep Learning CNN Score: 0.3390687108 (Confidence Match)
• Predicted ML Affinity  : 4.4657750130 (-log Kd scale)

--- Structural Pose Mode 4 ---
• Classical Vina Energy : -5.80543 kcal/mol
• Deep Learning CNN Score: 0.3220949471 (Confidence Match)
• Predicted ML Affinity  : 4.8985815048 (-log Kd scale)

--- Structural Pose Mode 5 ---
• Classical Vina Energy : -4.76018 kcal/mol
• Deep Learning CNN Score: 0.3219244182 (Confidence Match)
• Predicted ML Affinity  : 4.42954254

[01:35:26] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[01:35:26] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[01:35:26] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[01:35:26] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[01:35:26] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
